> **Executed output.** Exported from Databricks job `northbay-end-to-end` run 59476289597609, task `agent_evaluate_register` (task run 264795367540133), started 2026-11-08 08:46 UTC, duration 221s, result **SUCCESS**, serverless compute.


# 06 · Evaluate and register the resilience agent

The agent (LangGraph + `databricks-claude-sonnet-5-5`) has four tools:

| Tool | Backed by | Used for |
|---|---|---|
| `get_risk_ranking` | Lakebase `northbay_serving.stress_test_results` | "Where are we most exposed?" |
| `get_node_details` | Lakebase `northbay_serving.network_nodes / supply_links / sourcing_exposure` | Facts about a plant / supplier |
| `run_disruption_scenario` | TTR/TTS optimization on the live network, logged to Lakebase `app_state.scenario_runs` | What-if questions |
| `ask_genie` | Genie space over the gold tables | Descriptive analytics |

This notebook runs demo conversations, evaluates the agent with MLflow (LLM judges + a custom tool-routing
scorer), and logs / registers it in Unity Catalog. The same package runs inside the Databricks App.

In [ ]:
%pip install -r requirements.txt --quiet

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [ ]:
dbutils.library.restartPython()

In [ ]:
dbutils.widgets.text("catalog", "classic_stable_z57t9z_catalog")
dbutils.widgets.text("schema", "febar_supply_chain")
dbutils.widgets.text("warehouse_id", "")
dbutils.widgets.text("lakebase_instance", "febar-supply-chain-db")
catalog, schema = dbutils.widgets.get("catalog"), dbutils.widgets.get("schema")
warehouse_id, lakebase_instance = dbutils.widgets.get("warehouse_id"), dbutils.widgets.get("lakebase_instance")
fq = f"{catalog}.{schema}"
MODEL_NAME = f"{fq}.northbay_resilience_agent"

In [ ]:
import os
import sys

import mlflow

APP_DIR = os.path.abspath("../../app")
sys.path.append(APP_DIR)
from northbay_agent import config
from northbay_agent.lakebase import get_lakebase

config.GENIE_SPACE_ID = get_lakebase().query(
    "SELECT value FROM app_state.app_config WHERE key = 'genie_space_id'")[0]["value"]
user = spark.sql("SELECT current_user()").first()[0]
mlflow.set_experiment(f"/Users/{user}/northbay-resilience-agent")
mlflow.langchain.autolog()

from northbay_agent.agent import NorthBayResilienceAgent

AGENT = NorthBayResilienceAgent()
print("Genie space:", config.GENIE_SPACE_ID, "| LLM:", config.LLM_ENDPOINT)

2026/11/08 08:47:23 INFO mlflow.tracking.fluent: Experiment with name '/Users/ryuta.yoshimatsu@databricks.com/northbay-resilience-agent' does not exist. Creating a new experiment.
If you are using MLflow Tracing, consider storing your traces in Unity Catalog for unlimited storage (no 100,000 trace limit), fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/trace-unity-catalog
Genie space: 01f1db50b7331ec8bc8482c356a0c8cb | LLM: databricks-claude-sonnet-5-5


## Demo conversations

In [ ]:
from mlflow.types.agent import ChatAgentMessage


def chat(question: str):
    resp = AGENT.predict(messages=[ChatAgentMessage(role="user", content=question)])
    tools = [tc.function.name for m in resp.messages if m.tool_calls for tc in m.tool_calls]
    return resp.messages[-1].content, tools


for q in [
    "Where is NorthBay most exposed? Give me the top 3 suppliers and what they put at risk.",
    "Midwest Sweeteners (S207) just had a refinery explosion and expects 9 weeks to recover. What happens and what should we do?",
    "Which of our finished goods depend on a single supplier for any input?",
]:
    answer, tools = chat(q)
    print("=" * 100)
    print("USER:", q)
    print("TOOLS:", tools)
    print("AGENT:\n" + answer)

USER: Where is NorthBay most exposed? Give me the top 3 suppliers and what they put at risk.
TOOLS: ['get_risk_ranking']
AGENT:
NorthBay's three biggest exposures, from the latest portfolio stress test, are below. Each supplier is disrupted for its expected recovery time.

| Rank | Supplier | Disruption | TTR (weeks) | TTS (weeks) | Exposure gap (weeks) | Gross margin lost | Products at risk |
|---|---|---|---|---|---|---|---|
| 1 | Midwest Sweeteners (S207), Tier 2, IL, liquid sweetener & syrup base | Refinery explosion | 9 | 3.59 | 5.41 | **$2,987,053** | Morning Oats Cereal 18oz, Trail Granola Bar 6ct |
| 2 | Kerala Spice Traders (R301), Tier 3, IN, spices & dried herbs | Monsoon flooding | 12 | 5.26 | 6.74 | **$2,759,923** | Kettle Chips 8oz |
| 3 | FlavorCraft Blends (S201), Tier 2, IL, seasoning & flavor blends | Blending-line fire | 10 | 3.81 | 6.19 | **$2,418,816** | Kettle Chips 8oz |

**What this means**
- **Combined exposure:** the three suppliers together put about **$8.17M

## Evaluation

In [ ]:
from mlflow.entities import Feedback
from mlflow.genai.scorers import Guidelines, RelevanceToQuery, Safety, scorer

eval_data = [
    {"inputs": {"question": "Which suppliers would hurt us most if they went down?"},
     "expectations": {"expected_tools": ["get_risk_ranking"]}},
    {"inputs": {"question": "What happens if FlavorCraft Blends (S201) has a blending-line fire and is down for 10 weeks?"},
     "expectations": {"expected_tools": ["run_disruption_scenario"]}},
    {"inputs": {"question": "A hurricane takes out GulfCane Syrups (S208), Gulf Coast Resins (R303) and Bayou Sugar Mill (R309) for 8 weeks. Impact and actions?"},
     "expectations": {"expected_tools": ["run_disruption_scenario"]}},
    {"inputs": {"question": "If ClearPak Bottling (S203) is down 4 weeks, can we still serve Fizz Cola and SunValley Orange Drink?"},
     "expectations": {"expected_tools": ["run_disruption_scenario"]}},
    {"inputs": {"question": "What does BoxCraft Packaging supply and to which plants?"},
     "expectations": {"expected_tools": ["get_node_details", "ask_genie"]}},
    {"inputs": {"question": "How many weeks of finished-goods inventory does each plant hold?"},
     "expectations": {"expected_tools": ["ask_genie", "get_node_details"]}},
    {"inputs": {"question": "Which finished goods are single-sourced for sweetener?"},
     "expectations": {"expected_tools": ["ask_genie", "get_node_details"]}},
    {"inputs": {"question": "What's a good recipe for chocolate chip cookies?"},
     "expectations": {"expected_tools": []}},
]


@mlflow.trace(span_type="AGENT")
def predict_fn(question: str) -> str:
    resp = AGENT.predict(messages=[ChatAgentMessage(role="user", content=question)])
    return resp.messages[-1].content


@scorer
def tool_routing(trace, expectations) -> Feedback:
    """The agent must call at least one of the expected tools (or none for off-topic questions)."""
    called = sorted({s.name for s in trace.search_spans(span_type="TOOL")})
    expected = expectations["expected_tools"]
    ok = (not called) if not expected else bool(set(called) & set(expected))
    return Feedback(value=ok, rationale=f"called={called}, expected one of {expected}")


scorers = [
    tool_routing,
    RelevanceToQuery(),
    Safety(),
    Guidelines(name="business_ready", guidelines=[
        "If the question is about NorthBay's supply chain, the response quantifies impact with concrete numbers "
        "(USD and/or weeks) and avoids optimization jargon such as 'decision variables' or 'objective function'.",
        "If the question is off-topic (not about NorthBay's supply chain), the response politely declines.",
    ]),
    Guidelines(name="actionable", guidelines=[
        "If the question asks what happens or what to do about a disruption, the response gives at least two "
        "concrete recommended actions. Otherwise this guideline is satisfied.",
    ]),
]

results = mlflow.genai.evaluate(data=eval_data, predict_fn=predict_fn, scorers=scorers)
print({k: round(v, 3) if isinstance(v, float) else v for k, v in results.metrics.items()})

2026/11/08 08:48:48 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.
2026/11/08 08:48:48 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.
/databricks/python/lib/python3.12/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='content', input_value=[{'type': 'reasoning', 's...Eh97Q6S6IeJDBftGAE='}]}], input_type=list])
  return self.__pydantic_serializer__.to_python(
/databricks/python/lib/python3.12/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized valu

In [ ]:
tbl = results.tables["eval_results"] if hasattr(results, "tables") and "eval_results" in results.tables else None
if tbl is None:
    tbl = mlflow.search_traces(run_id=results.run_id)
cols = [c for c in tbl.columns if c.endswith("/value") or c in ("request", "inputs")]
print(tbl[cols].to_string(max_colwidth=90))

  actionable/value expected_tools/value  tool_routing/value business_ready/value safety/value relevance_to_query/value                                                                                    request
0              yes                 None                True                  yes          yes                      yes                      {'question': 'Which suppliers would hurt us most if they went down?'}
1              yes                 None                True                  yes          yes                      yes  {'question': 'What happens if FlavorCraft Blends (S201) has a blending-line fire and i...
2              yes                 None                True                  yes          yes                      yes  {'question': 'A hurricane takes out GulfCane Syrups (S208), Gulf Coast Resins (R303) a...
3              yes                 None                True                  yes          yes                      yes  {'question': 'If ClearPak Bottling (S203

## Log and register in Unity Catalog

In [ ]:
from mlflow.models.resources import DatabricksGenieSpace, DatabricksServingEndpoint, DatabricksSQLWarehouse

resources = [DatabricksServingEndpoint(endpoint_name=config.LLM_ENDPOINT),
             DatabricksGenieSpace(genie_space_id=config.GENIE_SPACE_ID),
             DatabricksSQLWarehouse(warehouse_id=warehouse_id)]
try:
    from mlflow.models.resources import DatabricksLakebase
    resources.append(DatabricksLakebase(database_instance_name=lakebase_instance))
except ImportError:
    pass

with mlflow.start_run(run_name="northbay-resilience-agent"):
    info = mlflow.pyfunc.log_model(
        name="agent",
        python_model=os.path.join(APP_DIR, "agent_model.py"),
        code_paths=[os.path.join(APP_DIR, "northbay_agent")],
        pip_requirements=[l.strip() for l in open("requirements.txt") if l.strip() and not l.startswith("#")],
        resources=resources,
        input_example={"messages": [{"role": "user", "content": "Which suppliers would hurt us most if they went down?"}]},
    )
    mlflow.log_metrics({k.replace("/", "_"): v for k, v in results.metrics.items() if isinstance(v, (int, float))})

mlflow.set_registry_uri("databricks-uc")
version = mlflow.register_model(info.model_uri, MODEL_NAME).version
mlflow.MlflowClient().set_registered_model_alias(MODEL_NAME, "champion", version)
print(f"registered {MODEL_NAME} version {version} (alias: champion)")

🔗 View Logged Model at: https://adb-7405610715852579.19.azuredatabricks.net/ml/experiments/2427419569164653/models/m-eb26151fcb664b3c9917ce208f2694c8?o=7405610715852579
2026/11/08 08:49:35 INFO mlflow.pyfunc: Predicting on input example to validate output
Successfully registered model 'classic_stable_z57t9z_catalog.febar_supply_chain.northbay_resilience_agent'.
🔗 Created version '1' of model 'classic_stable_z57t9z_catalog.febar_supply_chain.northbay_resilience_agent': https://adb-7405610715852579.19.azuredatabricks.net/explore/data/models/classic_stable_z57t9z_catalog/febar_supply_chain/northbay_resilience_agent/version/1?o=7405610715852579
registered classic_stable_z57t9z_catalog.febar_supply_chain.northbay_resilience_agent version 1 (alias: champion)
